### Activation Function 
Activation functions are mathematical equations that determine the output of a neural network node. They introduce non-linearity into the network, enabling it to learn complex patterns beyond simple linear relationships.

Without activation function will act like a linear regression
Common Activation function
- ReLU (Rectified Linear Unit)
- Sigmoid (Logistic)
- Tanh
- Softmax

In [1]:
import torch
import torch.nn as nn
import numpy as np

### 1. ReLU
$$ReLU(x) = max(0, x) $$
Derivative
$$f'(x) = 
\begin{cases} 
0 & \text{if } x < 0 \\
1 & \text{if } x > 0 
\end{cases}$$

Characteristics:
- Reduce Vanishing Gradient
- Computationally efficient

When to Use
- Default choice in hidden layer

Downside
- Dying Relu

In [ ]:
class ActivationReLU:
    def forward(self, x):
        self.x = x
        return np.maximum(x,0)

    def backward(self, d_out):
        # dL/ dx = (dL / d d_out) * (d d_out / dx) 
        d_input = d_out.copy()
        d_input[self.x <= 0] = 0
        return d_input

x = np.random.normal(size=(2,3))
relu_np = ActivationReLU()
out_np = relu_np.forward(x)
d_out = np.ones_like(x)
grad_np = relu_np.backward(d_out)

relu_nn = nn.ReLU()
x_nn = torch.tensor(x,requires_grad=True)
out_nn = relu_nn(x_nn)
out_nn.backward(torch.ones_like(out_nn))
grad_nn = x_nn.grad

assert np.allclose(out_np,out_nn.detach().numpy())
assert np.allclose(grad_nn,grad_np)


### 2. Sigmoid
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
where $$z = w_1x_1 + w_2x_2 + b$$
Range = [0,1]
Derivative
$${\sigma ′}(z) = \sigma(z) (1 - \sigma(z))$$


Characteristics:
- Sooth Gradient
- Probability Interpretation(value between 0 and 1)

When to Use
- Output layer binary classification 

Downside
- Vanishing Gradient

In [ ]:
class SigmoidNP:
    def forward(self,z:np.ndarray):
        self.z = z
        return 1 / (1 + np.exp(-z))

    def backward(self,d_out):
        s = self.forward(self.z)
        return d_out * s * (1 - s)

x = np.random.normal(size=(1,3))
sigmoid_np = SigmoidNP()
out_np = sigmoid_np.forward(x)
d_out = np.ones_like(x)
grad_np = sigmoid_np.backward(d_out)

sigmoid_nn = nn.Sigmoid()
x_nn = torch.tensor(x,requires_grad=True)
out_nn = sigmoid_nn(x_nn)
out_nn.backward(torch.ones_like(out_nn))
grad_nn = x_nn.grad

assert np.allclose(out_np,out_nn.detach().numpy())
assert np.allclose(grad_nn,grad_np)


### 3. Tanh
$$\tanh(z) = \frac{\sinh(x)}{\cosh(x)} = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$
Range = [-1,1]
Derivative
$$\bar{\tanh}(x) =  (1 - \tanh^2(x))$$


Characteristics:
- Zero centric

When to Use
- RNN  

Downside
- Vanishing Gradient

In [11]:
class TanHNP:
    def forward(self,z:np.ndarray):
        self.z = z
        return (np.exp(z) - np.exp(-z)) / (np.exp(z) + np.exp(-z))

    def backward(self,d_out):
        return d_out * (1 - np.square(self.forward(self.z)))

x = np.random.normal(size=(1,3))
tanh_np = TanHNP()
out_np = tanh_np.forward(x)
d_out = np.ones_like(x)
grad_np = tanh_np.backward(d_out)

tanh_nn = nn.Tanh()
x_nn = torch.tensor(x,requires_grad=True)
out_nn = tanh_nn(x_nn)
out_nn.backward(torch.ones_like(out_nn))
grad_nn = x_nn.grad

assert np.allclose(out_np,out_nn.detach().numpy())
assert np.allclose(grad_nn,grad_np)


### 4. Softmax
$$softmax(z_i) = \frac{e^{z_i}}{\sum{e^{z_j}}}$$

Derivative
$$\frac{\partial \text{softmax}(z_i)}{\partial z_j} = \text{softmax}(z_i)(\delta_{ij} - \text{softmax}(z_j))$$

Characteristics:
- Convert raw vector score into probability distrubtion

When to Use
- Multiclass classification output layer  

**Numerical stability** - In Softmax function if input are large, then value of exponential function will be extramly large leading to `NaN`.
To prevent overflow issue we subtract input with maximum value of inputs.
$${\sum_j e^{z_j}} = \frac{e^{z_i - m}}{\sum_j e^{z_j - m}}$$
where $m = \max(x)$

In [ ]:
class SoftmaxNP:
    def forward(self,x:np.ndarray):
        self.x = x
        exp_x = np.exp(x - np.max(x, axis=1,keepdims=True))
        self.out = exp_x / np.sum(exp_x , axis=1, keepdims=True)
        return self.out

    def backward(self,d_out):
        s = self.out
        dot = np.sum(d_out * s, axis=1, keepdims=True)
        return s * (d_out - dot)

softmax_np = SoftmaxNP()
x = np.random.normal(size=(2, 3))
out_np = softmax_np.forward(x)
d_out = np.random.normal(size=(2, 3))
grad_np = softmax_np.backward(d_out)

softmax_nn = nn.Softmax(dim=1)         
x_nn = torch.tensor(x, requires_grad=True)
out_nn = softmax_nn(x_nn)
out_nn.backward(torch.tensor(d_out))
grad_nn = x_nn.grad.numpy()

assert np.allclose(out_np,out_nn.detach().numpy())
assert np.allclose(grad_nn,grad_np)
